# 📄🔍 Del PDF al RAG: documentos con estructura, con BigQuery y Document AI

**Curso práctico · ~90 minutos · Google Colab + BigQuery + Document AI**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noelserdna/colab-gcp-ia/blob/main/curso_rag_pdf_polizas_bigquery.ipynb)

Trabajamos para **Peñalara Seguros, S.A.**. El equipo de atención al cliente tiene un problema muy concreto: cuando un cliente pregunta *"¿esto me lo cubre la póliza?"*, la respuesta está en un **condicionado general de decenas de páginas**, lleno de tablas, cláusulas numeradas y letra pequeña. Buscar a mano es lento; equivocarse es caro.

Vamos a construir un RAG que responda **citando la póliza, la página y la cláusula exacta**.

> **Este curso es la continuación de [*De la bandeja de entrada al RAG*](https://colab.research.google.com/github/noelserdna/colab-gcp-ia/blob/main/curso_rag_emails_bigquery_v2.ipynb).** Damos por sabidos los fundamentos: qué es un embedding, qué es `task_type`, cómo funciona `VECTOR_SEARCH` y qué es el patrón RAG. Aquí **no los repetimos**: nos centramos en lo que **cambia** cuando la fuente ya no es texto, sino un PDF.


## 🎒 Kit de supervivencia (si llegas sin haber hecho el curso de emails)

Este curso es la **segunda parte** de una serie, así que da por vistos algunos conceptos. Si vienes del [curso de RAG sobre emails](https://colab.research.google.com/github/noelserdna/colab-gcp-ia/blob/main/curso_rag_emails_bigquery_v2.ipynb), **puedes saltarte este recuadro**. Si es tu primera vez, aquí tienes lo justo para no perderte — cinco ideas, treinta segundos cada una:

- **Embedding** — convertir un texto en una lista de números (un *vector*) que captura su **significado**. Dos textos que quieren decir lo mismo tienen vectores parecidos, *aunque no compartan ni una palabra*. Es lo que permite buscar por **sentido** en vez de por coincidencia exacta de palabras.
- **Distancia / similitud** — si cada texto es un punto en un espacio, los textos parecidos caen **cerca**. "Buscar" es, literalmente, encontrar los puntos más próximos a la pregunta. Aquí medimos esa cercanía con *distancia coseno*.
- **Chunk (fragmento)** — un documento entero es demasiado grande para vectorizarlo de golpe, así que se **trocea** en fragmentos y se vectoriza cada uno por separado. **Cómo** se trocea es justo el tema central de este curso.
- **`task_type`** — al pedir un embedding le indicas para qué es: **`RETRIEVAL_DOCUMENT`** cuando vectorizas los documentos, **`RETRIEVAL_QUERY`** cuando vectorizas la pregunta del usuario. Usar el correcto mejora bastante la búsqueda.
- **RAG (Retrieval-Augmented Generation)** — el patrón que montamos, en tres pasos: **(1) recuperar** los fragmentos relevantes, **(2) aumentar** el prompt pegándoselos como contexto, y **(3) generar** la respuesta usando *solo* ese contexto. Así el LLM responde con **tus** documentos y puede **citar** de dónde sacó cada cosa, en lugar de inventar.

> Con esto te sobra para seguir. Cuando más abajo veas `ML.GENERATE_EMBEDDING`, `VECTOR_SEARCH` o el ciclo *recuperar → aumentar → generar*, ya sabrás qué son. El resto lo explicamos sobre la marcha.


---
## ⚖️ La pregunta que abre el curso: ¿qué cambia respecto a los emails?

En el curso anterior, la "materia prima" era cómoda: un email **ya es texto**. Venía en una columna `body`, con sus metadatos (`from`, `fecha`, `categoria`) en columnas de al lado. El trabajo empezaba directamente en el chunking.

Un PDF no te regala nada de eso. **Un PDF no es un formato de datos: es un formato de presentación.** Por dentro no hay párrafos ni secciones — hay instrucciones de dibujo: *"pon este glifo en la coordenada (x=72, y=531)"*. La estructura que tú ves (el título, la tabla, la columna) **existe solo en tu cabeza**, reconstruida a partir de la posición de las cosas en la página.

| | 📧 Emails (curso anterior) | 📄 PDFs (este curso) |
|---|---|---|
| **Dónde vive** | Filas de una tabla | Archivos binarios en Cloud Storage |
| **El texto…** | ya viene dado (`body`) | hay que **extraerlo** de un formato visual |
| **Estructura** | columnas (`from`, `fecha`) | **layout**: páginas, columnas, tablas, encabezados |
| **Unidad natural** | 1 email ≈ 1 idea (~200 palabras) | 1 póliza = decenas de páginas jerárquicas |
| **Metadatos** | vienen dados | hay que **derivarlos** (página, sección, cláusula) |
| **Chunking** | cortar por caracteres bastaba | cortar por caracteres **destruye** tablas y jerarquía |
| **Citar la fuente** | "el email msg-0042" | **"Hogar Plus, pág. 5, cláusula 4.1.a"** ← legalmente crítico |

### 🧠 El nuevo desafío, en una frase

> En los emails, el reto era **encontrar** el texto relevante.
> En los PDFs, el reto es que **el texto relevante no existe todavía**: hay que fabricarlo, y fabricarlo **sin perder de dónde venía**.


### 🎯 El caso que lo resume todo: cobertura vs. exclusión

Aquí está el corazón de este curso. Fíjate en estos dos fragmentos del condicionado de **Hogar Plus**:

> **Página 4** — *"Daños por agua por rotura accidental de conducciones — 50.000 € — franquicia 150 €"*

> **Página 5** — *"Los daños causados por humedades, condensación o **filtraciones** a través de muros, fachadas, terrazas o cubiertas, **aun cuando sean consecuencia de lluvia**, nieve o granizo."*

Un cliente pregunta: **"¿me cubre los daños del agua de lluvia que me ha entrado en casa?"**

Los dos fragmentos hablan de *daños*, de *agua*, de *la póliza de hogar*. Para un buscador semántico **los dos son relevantísimos**: comparten casi todo el significado. Pero uno vive bajo el título **"3. COBERTURAS"** y el otro bajo **"4. EXCLUSIONES"**.

**Uno dice que sí y el otro dice que no.**

Si troceamos el PDF cortando cada 500 caracteres —como hacíamos con los emails— el fragmento de la página 5 llega al modelo **huérfano**: un párrafo sobre daños por agua, sin el título que lo condena. El LLM leerá *"daños... agua... lluvia..."*, verá que encaja con la pregunta, y responderá alegremente **"sí, está cubierto"**.

Acabas de construir un sistema que **le miente a tu cliente sobre su póliza**. Y lo hace con una seguridad impecable.

> 🧭 **La tesis de este curso:** en documentos estructurados, un chunk **sin su contexto jerárquico no es información: es una trampa**. Todo lo que hacemos a continuación existe para evitar exactamente este error.


---
## 🛠️ Cómo lo afrontamos (y por qué así)

Hay dos formas de sacar el texto de un PDF:

**Opción A — Extraerlo nosotros** (`pypdf`, `pdfplumber`). Barato, rápido, sin dependencias externas. Te devuelve un río de texto plano: **pierdes la jerarquía, y las tablas se convierten en espagueti**. Además, si el PDF es un **escaneo** (una foto del papel), no hay texto que extraer: te devuelve vacío.

**Opción B — Document AI Layout Parser** (la que usaremos). Un servicio de Google que **entiende la página**: distingue títulos de párrafos, detecta tablas, ordena las columnas, hace OCR si hace falta, y —lo decisivo para nosotros— **trocea el documento respetando su estructura** y puede **inyectar en cada chunk los títulos de los que cuelga**.

Esa última capacidad (`include_ancestor_headings`) es la que convierte el fragmento huérfano de la página 5 en:

```
# 4. EXCLUSIONES
## 4.1 Daños por agua no cubiertos
Los daños causados por humedades, condensación o filtraciones...
```

Ahora el chunk **se defiende solo**. Cuando llegue al LLM, el modelo verá el título `4. EXCLUSIONES` pegado al texto y responderá lo correcto. **No es magia: es contexto.**

### La arquitectura

```
   PDFs (pólizas)
        │  gsutil cp
        ▼
  ┌─────────────┐
  │ Cloud Storage│   los PDF viven aquí (BigQuery no guarda binarios)
  └──────┬──────┘
         │  object table  (tabla externa: 1 fila = 1 archivo)
         ▼
  ┌──────────────────────────────────────┐
  │ BigQuery                             │
  │   ML.PROCESS_DOCUMENT  ──────────────┼──► Document AI (Layout Parser)
  │      ↓ chunks con página + headings  │      · entiende layout
  │   ML.GENERATE_EMBEDDING ─────────────┼──► Vertex AI (embeddings)
  │      ↓ vectores                      │
  │   VECTOR_SEARCH  → contexto          │
  └──────────────┬───────────────────────┘
                 ▼
           Gemini responde CITANDO póliza + página + cláusula
```

> 🎓 **Fíjate en lo que NO cambia:** de `ML.GENERATE_EMBEDDING` hacia abajo, esto es **exactamente** el mismo pipeline del curso de emails. Todo el trabajo nuevo está *antes*: en convertir un archivo binario en chunks que conserven su contexto. **Ese es el 90% del trabajo real de un RAG documental.**


### 📋 Requisitos (¡verifícalos ANTES de la clase!)

- Un **proyecto de Google Cloud** con **facturación activa**.
  > ⚠️ **Importante y distinto al curso anterior:** **Document AI no tiene capa gratuita**. Cuesta **$10 por cada 1.000 páginas**. Este cuaderno procesa **24 páginas ≈ $0,24**. Es calderilla, y los $300 de crédito de prueba lo cubren de sobra — pero **el sandbox de BigQuery sin tarjeta no sirve**: sin billing no podrás crear el procesador.
- Permisos en el proyecto para: crear datasets y conexiones en BigQuery, crear buckets, crear un procesador de Document AI y conceder roles IAM.
- Haber hecho (o al menos entendido) el **curso de RAG sobre emails**.

### Agenda

| # | Bloque | ⏱️ |
|---|--------|----|
| 0 | Setup | 5 min |
| 1 | Fabricar las pólizas en PDF | 5 min |
| 2 | Los PDF a Cloud Storage + object table | 10 min |
| 3 | Document AI: procesador y modelo remoto | 10 min |
| 4 | `ML.PROCESS_DOCUMENT`: del PDF a chunks con estructura | 15 min |
| 5 | **El contraste**: chunking naive vs. Layout Parser | 10 min |
| 6 | Embeddings y búsqueda (rápido: ya lo sabes) | 10 min |
| 7 | RAG con **citas verificables** | 15 min |
| 8 | Versionado de documentos e incremental | 10 min |
| 9 | Evaluación, ejercicios y limpieza | 10 min |


---
# 0 · Setup ⏱️ ~5 min

▶️ **Qué hace esta celda:** instala las librerías. Respecto al curso anterior sobran `faker` (ya no generamos emails) y sobra `pdfplumber`… pero añadimos **`reportlab`** para *fabricar* los PDF y **`pypdf`** para la comparación del bloque 5.

In [1]:
%pip install --quiet --upgrade google-cloud-bigquery google-cloud-storage google-genai pandas db-dtypes reportlab pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 733.7 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 21.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.


▶️ **Qué hace esta celda:** te autentica, fija el proyecto y habilita las APIs. **Novedad respecto al curso de emails:** además de BigQuery y Vertex AI, necesitamos **`documentai`** (el parser) y **`storage`** (los PDF viven en un bucket).

> ⚠️ `LOCATION` debe ser **`US`** o **`EU`**. No es un capricho: `ML.PROCESS_DOCUMENT` **solo existe en esas dos multi-regiones**, y el dataset, la conexión y el procesador **deben estar los tres en la misma**.

In [5]:
from google.colab import auth
auth.authenticate_user()
print("✅ Autenticado")

# 👇 EDITA ESTO con tu proyecto
PROJECT_ID = "codecrypto-ai"  # @param {type:"string"}
LOCATION   = "US"               # BigQuery: US o EU (obligatorio para ML.PROCESS_DOCUMENT)
DATASET    = "rag_polizas"
BUCKET     = f"{PROJECT_ID}-polizas-rag"   # los PDF vivirán aquí
LOC_DOCAI  = LOCATION.lower()              # Document AI usa 'us' / 'eu' en minúscula

!gcloud config set project {PROJECT_ID} --quiet
# Habilitamos las APIs necesarias (idempotente)
!gcloud services enable bigquery.googleapis.com bigqueryconnection.googleapis.com \
    aiplatform.googleapis.com documentai.googleapis.com storage.googleapis.com --quiet
print("✅ APIs habilitadas")

✅ Autenticado
[environment: untagged] Read more to tag: g.co/cloud/project-env-tag.
Updated property [core/project].
Operation "operations/acat.p2-540389217585-bf41bd16-1883-4574-824f-85ecf8e20b8c" finished successfully.
✅ APIs habilitadas


▶️ **Qué hace esta celda:** crea el cliente de BigQuery y el dataset (igual que en el curso anterior).

In [6]:
from google.cloud import bigquery

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)

ds = bigquery.Dataset(f"{PROJECT_ID}.{DATASET}")
ds.location = LOCATION
client.create_dataset(ds, exists_ok=True)
print(f"✅ Dataset listo: {PROJECT_ID}.{DATASET}")

✅ Dataset listo: codecrypto-ai.rag_polizas


> 🚩 **CHECKPOINT 1** — Todos deberíais ver `✅ Dataset listo`. Si falla aquí, casi siempre es el `PROJECT_ID` mal escrito o el proyecto **sin facturación activa**.

---
# 1 · Fabricamos las pólizas en PDF ⏱️ ~5 min

En el curso anterior generamos emails sintéticos con `faker`. Aquí generamos **condicionados de seguro en PDF** con `reportlab`.

No es relleno: los PDF están **diseñados para tener los problemas reales** de un documento de seguros, y cada elemento tiene un porqué didáctico.

| Elemento del PDF | Para qué está ahí |
|---|---|
| Numeración jerárquica (`4.1.a`) | permitir **citar** la cláusula exacta |
| **Tablas** de coberturas y franquicias | ver cómo las destroza el chunking naive |
| Sección `4. EXCLUSIONES` con texto **parecido** al de coberturas | 🎯 **el gotcha del curso** |
| Letra pequeña (6,5 pt) | la cláusula limitativa que nadie lee |
| Pie *"Página X de Y"* | lo captura `pageFooters` del Layout Parser |
| ~8 páginas por póliza | por debajo del límite del parser (15 páginas) |


▶️ **Qué hace esta celda:** define los estilos y la plantilla del documento. Lo relevante didácticamente: `SMALL` es la **letra pequeña** (6,5 pt) y `PolizaDoc` pinta el **encabezado y el pie de página** — ese pie es un metadato que luego recuperaremos del parser.

In [7]:
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.platypus import (BaseDocTemplate, PageTemplate, Frame, Paragraph,
                                Spacer, Table, TableStyle, PageBreak)
from reportlab.lib.enums import TA_JUSTIFY, TA_CENTER

ASEGURADORA = "Peñalara Seguros, S.A."

ss = getSampleStyleSheet()
H1 = ParagraphStyle("H1x", parent=ss["Heading1"], fontSize=15, spaceAfter=10,
                    textColor=colors.HexColor("#1a3d5c"))
H2 = ParagraphStyle("H2x", parent=ss["Heading2"], fontSize=12, spaceBefore=10,
                    spaceAfter=6, textColor=colors.HexColor("#2c5f8a"))
H3 = ParagraphStyle("H3x", parent=ss["Heading3"], fontSize=10.5, spaceBefore=8,
                    spaceAfter=4, textColor=colors.HexColor("#444444"))
BODY = ParagraphStyle("BODYx", parent=ss["BodyText"], fontSize=9.5, leading=13,
                      alignment=TA_JUSTIFY, spaceAfter=5)
# 👇 la famosa "letra pequeña": legalmente válida, visualmente hostil
SMALL = ParagraphStyle("SMALLx", parent=BODY, fontSize=6.5, leading=8.5,
                       textColor=colors.HexColor("#555555"))
TITLE = ParagraphStyle("TITLEx", parent=ss["Title"], fontSize=22,
                       textColor=colors.HexColor("#1a3d5c"))
CENTER = ParagraphStyle("CENTERx", parent=BODY, alignment=TA_CENTER)

def _tabla(data, col_widths):
    t = Table(data, colWidths=col_widths, repeatRows=1)
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#1a3d5c")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("FONTSIZE", (0, 0), (-1, -1), 8),
        ("GRID", (0, 0), (-1, -1), 0.4, colors.grey),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#eef3f8")]),
    ]))
    return t

class PolizaDoc(BaseDocTemplate):
    """Documento con encabezado y pie 'Página X de Y' en cada página."""
    def __init__(self, filename, producto, codigo, **kw):
        super().__init__(filename, pagesize=A4, **kw)
        self.producto, self.codigo = producto, codigo
        frame = Frame(2.2*cm, 2.2*cm, A4[0]-4.4*cm, A4[1]-4.4*cm, id="n")
        self.addPageTemplates([PageTemplate(id="all", frames=[frame], onPage=self._decorar)])

    def _decorar(self, canvas, doc):
        canvas.saveState()
        canvas.setFont("Helvetica", 7)
        canvas.setFillColor(colors.HexColor("#777777"))
        canvas.drawString(2.2*cm, A4[1]-1.5*cm, f"{ASEGURADORA} · {self.producto}")
        canvas.drawRightString(A4[0]-2.2*cm, A4[1]-1.5*cm, f"Condicionado {self.codigo}")
        canvas.line(2.2*cm, A4[1]-1.65*cm, A4[0]-2.2*cm, A4[1]-1.65*cm)
        canvas.line(2.2*cm, 1.9*cm, A4[0]-2.2*cm, 1.9*cm)
        canvas.drawCentredString(A4[0]/2.0, 1.4*cm, f"Página {doc.page}")
        canvas.restoreState()

print("✅ Estilos y plantilla definidos")

✅ Estilos y plantilla definidos


▶️ **Qué hace esta celda:** la función que arma el documento sección a sección. Mira el orden: **`3. COBERTURAS`** (con su tabla) y justo después **`4. EXCLUSIONES`**. Esa vecindad es deliberada — es la que va a poner a prueba nuestro RAG.

In [8]:
def construir_poliza(path, producto, codigo, version, coberturas, exclusiones, franquicias):
    story = []
    # ── Portada ──
    story += [Spacer(1, 4*cm), Paragraph(ASEGURADORA, CENTER), Spacer(1, 1*cm),
              Paragraph(producto, TITLE), Spacer(1, 0.6*cm),
              Paragraph("Condiciones Generales", CENTER), Spacer(1, 0.3*cm),
              Paragraph(f"Código de condicionado: <b>{codigo}</b> · Versión {version}", CENTER),
              PageBreak()]

    # ── 1. Definiciones ──
    story += [Paragraph("1. DEFINICIONES", H1),
              Paragraph("A efectos del presente contrato, se entiende por:", BODY)]
    for num, term, txt in [
        ("1.1", "Asegurado", "Persona física o jurídica titular del interés asegurado y sobre la que recaen las consecuencias económicas del siniestro."),
        ("1.2", "Tomador", "Persona que suscribe el contrato con el Asegurador y a quien corresponden las obligaciones derivadas del mismo."),
        ("1.3", "Siniestro", "Todo hecho cuyas consecuencias estén total o parcialmente cubiertas por las garantías de esta póliza."),
        ("1.4", "Franquicia", "Cantidad que queda a cargo del Asegurado en cada siniestro y que se deduce de la indemnización."),
        ("1.5", "Suma asegurada", "Límite máximo de indemnización por siniestro y anualidad de seguro."),
    ]:
        story += [Paragraph(f"{num} {term}", H3), Paragraph(txt, BODY)]

    # ── 2. Objeto ──
    story += [PageBreak(), Paragraph("2. OBJETO DEL SEGURO", H1),
              Paragraph(f"El Asegurador garantiza, dentro de los límites del presente condicionado, "
                        f"las consecuencias económicas de los riesgos descritos en la sección 3, hasta "
                        f"las sumas fijadas en las Condiciones Particulares de la póliza {producto}.", BODY),
              Paragraph("2.1 Ámbito territorial", H3),
              Paragraph("Las garantías surten efecto en el territorio español, salvo indicación expresa "
                        "en contrario en las Condiciones Particulares.", BODY),
              Paragraph("2.2 Ámbito temporal", H3),
              Paragraph("Quedan cubiertos los siniestros ocurridos durante la vigencia de la póliza y "
                        "declarados conforme a los plazos de la sección 6.", BODY)]

    # ── 3. Coberturas (CON TABLA) ──
    story += [PageBreak(), Paragraph("3. COBERTURAS", H1),
              Paragraph("Quedan cubiertas las siguientes garantías, con los límites indicados:", BODY),
              Spacer(1, 0.3*cm),
              _tabla([["Garantía", "Límite por siniestro", "Franquicia"]] + coberturas,
                     [7.5*cm, 4.5*cm, 3.5*cm]), Spacer(1, 0.4*cm)]
    for i, (gar, lim, fr) in enumerate(coberturas, start=1):
        story += [Paragraph(f"3.{i} {gar}", H3),
                  Paragraph(f"Se garantiza el pago de la indemnización por los daños directos "
                            f"ocasionados por {gar.lower()}, hasta el límite de {lim} por siniestro, "
                            f"con una franquicia de {fr}. La cobertura opera siempre que el hecho "
                            f"causante sea súbito, accidental e imprevisto para el Asegurado.", BODY)]

    # ── 4. Exclusiones (🎯 el gotcha) ──
    story += [PageBreak(), Paragraph("4. EXCLUSIONES", H1),
              Paragraph("<b>Con carácter general, y salvo pacto expreso en contrario, quedan "
                        "EXCLUIDOS de toda cobertura:</b>", BODY)]
    for i, (titulo, items) in enumerate(exclusiones, start=1):
        story += [Paragraph(f"4.{i} {titulo}", H2)]
        for letra, txt in zip("abcdefghij", items):
            story += [Paragraph(f"4.{i}.{letra}) {txt}", BODY)]
    story += [Spacer(1, 0.3*cm),
              Paragraph("Las exclusiones recogidas en la presente sección han sido específicamente "
                        "aceptadas por el Tomador mediante su firma en las Condiciones Particulares, "
                        "conforme al artículo 3 de la Ley 50/1980, de Contrato de Seguro, que exige "
                        "que las cláusulas limitativas de los derechos del Asegurado se destaquen de "
                        "modo especial y sean expresamente aceptadas por escrito.", SMALL)]

    # ── 5. Franquicias (otra tabla) ──
    story += [PageBreak(), Paragraph("5. FRANQUICIAS", H1),
              Paragraph("Se aplicarán las siguientes franquicias por modalidad:", BODY),
              Spacer(1, 0.3*cm),
              _tabla([["Modalidad", "Franquicia general", "Franquicia específica"]] + franquicias,
                     [6*cm, 4.75*cm, 4.75*cm])]

    # ── 6. Siniestros ──
    story += [PageBreak(), Paragraph("6. DECLARACIÓN Y TRAMITACIÓN DE SINIESTROS", H1),
              Paragraph("6.1 Plazo de comunicación", H3),
              Paragraph("El Tomador deberá comunicar el siniestro al Asegurador en el plazo máximo de "
                        "<b>siete (7) días</b> desde que tuviera conocimiento del mismo.", BODY),
              Paragraph("6.2 Documentación exigible", H3),
              Paragraph("Deberá aportarse: declaración del siniestro, acreditación de la titularidad "
                        "del bien, presupuesto o factura de reparación y, cuando proceda, atestado.", BODY),
              Paragraph("6.3 Peritación", H3),
              Paragraph("En caso de desacuerdo sobre la valoración, cada parte designará un perito. De "
                        "persistir la discrepancia, se designará un tercer perito de común acuerdo.", BODY),
              Paragraph("6.4 Pago de la indemnización", H3),
              Paragraph("El Asegurador abonará la indemnización en el plazo de cuarenta (40) días desde "
                        "la recepción de la declaración del siniestro.", BODY)]

    # ── 7. Prima ──
    story += [PageBreak(), Paragraph("7. PRIMA, DURACIÓN Y RENOVACIÓN", H1),
              Paragraph("7.1 Pago de la prima", H3),
              Paragraph("La prima es anual y pagadera por anticipado.", BODY),
              Paragraph("7.2 Impago y suspensión", H3),
              Paragraph("En caso de impago de la segunda o sucesivas primas, la cobertura quedará "
                        "<b>suspendida un mes después</b> del día de su vencimiento.", BODY),
              Paragraph("7.3 Duración y prórroga", H3),
              Paragraph("El contrato se prorrogará tácitamente por periodos anuales, salvo oposición "
                        "notificada con <b>un (1) mes</b> de antelación por el Tomador o <b>dos (2) "
                        "meses</b> por el Asegurador.", BODY)]

    PolizaDoc(path, producto, codigo).multiBuild(story)
    return path

print("✅ Constructor de pólizas definido")

✅ Constructor de pólizas definido


▶️ **Qué hace esta celda:** el catálogo de productos y la generación. Lee con atención las **exclusiones de Hogar Plus**: `4.1.a` habla de *filtraciones... aun cuando sean consecuencia de lluvia* y `4.1.d` de *agua de lluvia que penetre por ventanas*. Compáralas con la cobertura `3.2` (*daños por agua por rotura de conducciones*). **Un humano con prisa las confunde. Un RAG mal hecho, también.**

In [9]:
import os

POLIZAS = [
    dict(
        path="HOGAR_PLUS.pdf", producto="Hogar Plus", codigo="HP-2026-01", version="3.2",
        coberturas=[
            ["Incendio, rayo y explosión", "300.000 €", "Sin franquicia"],
            ["Daños por agua por rotura accidental de conducciones", "50.000 €", "150 €"],
            ["Robo y expoliación en el interior de la vivienda", "30.000 €", "150 €"],
            ["Rotura de cristales y vitrocerámica", "3.000 €", "Sin franquicia"],
            ["Responsabilidad civil familiar", "150.000 €", "300 €"],
            ["Fenómenos atmosféricos (viento, pedrisco, nieve)", "100.000 €", "300 €"],
        ],
        exclusiones=[
            ("Daños por agua no cubiertos", [
                "Los daños causados por <b>humedades, condensación o filtraciones</b> a través de muros, "
                "fachadas, terrazas o cubiertas, aun cuando sean consecuencia de lluvia, nieve o granizo.",
                "Los daños derivados de <b>falta de mantenimiento</b> de las conducciones, así como la "
                "corrosión, el óxido o el desgaste paulatino de tuberías.",
                "El coste de <b>localización y reparación de la avería</b> cuando no se haya producido "
                "daño material indemnizable.",
                "Los daños por <b>agua de lluvia que penetre por ventanas, puertas o huecos dejados "
                "abiertos</b> o defectuosamente cerrados por el Asegurado.",
            ]),
            ("Exclusiones generales", [
                "Los daños causados con dolo o culpa grave del Asegurado.",
                "Los daños derivados de <b>vicio propio o defecto de construcción</b> preexistente.",
                "Los daños calificados como catástrofe nacional o cubiertos por el <b>Consorcio de "
                "Compensación de Seguros</b>.",
                "Los daños en <b>viviendas deshabitadas</b> más de 60 días consecutivos.",
            ]),
        ],
        franquicias=[["Vivienda habitual", "150 €", "300 € en RC familiar"],
                     ["Segunda residencia", "300 €", "600 € en daños por agua"],
                     ["Vivienda en alquiler", "300 €", "600 € en robo"]],
    ),
    dict(
        path="AUTO_TODO_RIESGO.pdf", producto="Auto Todo Riesgo", codigo="AT-2026-04", version="2.1",
        coberturas=[
            ["Responsabilidad civil obligatoria", "Ilimitada (legal)", "Sin franquicia"],
            ["Daños propios por colisión o vuelco", "Valor venal + 20%", "300 €"],
            ["Robo total o parcial del vehículo", "Valor venal", "300 €"],
            ["Incendio del vehículo", "Valor venal", "Sin franquicia"],
            ["Lunas (parabrisas, laterales y trasera)", "Sin límite", "Sin franquicia"],
            ["Asistencia en viaje desde kilómetro 0", "Incluida", "Sin franquicia"],
        ],
        exclusiones=[
            ("Circunstancias del conductor", [
                "Siniestros conduciendo bajo <b>influencia de bebidas alcohólicas</b>, drogas o estupefacientes.",
                "Siniestros cuando el conductor <b>carezca de permiso de conducción</b> en vigor.",
                "Siniestros en <b>carreras, apuestas o pruebas deportivas</b> y sus entrenamientos.",
            ]),
            ("Uso del vehículo", [
                "El uso como <b>autoescuela, alquiler sin conductor, taxi o VTC</b>, salvo declaración expresa.",
                "El transporte de <b>mercancías peligrosas</b> o de más ocupantes de los autorizados.",
                "Los daños circulando por <b>vías no aptas</b> para la circulación o fuera de calzada.",
            ]),
            ("Daños no indemnizables", [
                "El <b>desgaste, uso o defecto de conservación</b> de las piezas.",
                "Los daños <b>exclusivamente estéticos</b> que no afecten a la seguridad.",
                "La <b>depreciación</b> del vehículo tras la reparación.",
            ]),
        ],
        franquicias=[["Conductor > 25 años y > 2 años de carné", "300 €", "Sin franquicia en lunas"],
                     ["Conductor novel (< 2 años de carné)", "600 €", "600 € en daños propios"],
                     ["Conductor ocasional no declarado", "900 €", "900 € en daños propios"]],
    ),
    dict(
        path="SALUD_FAMILIAR.pdf", producto="Salud Familiar", codigo="SF-2026-02", version="1.4",
        coberturas=[
            ["Medicina primaria y especialidades", "Sin límite", "Sin franquicia"],
            ["Pruebas diagnósticas (analítica, radiología)", "Sin límite", "Sin franquicia"],
            ["Hospitalización y cirugía en centros concertados", "Sin límite", "Sin franquicia"],
            ["Urgencias 24 h en cuadro médico", "Sin límite", "Sin franquicia"],
            ["Fisioterapia y rehabilitación", "30 sesiones/año", "10 € por sesión"],
            ["Psicología clínica", "20 sesiones/año", "15 € por sesión"],
        ],
        exclusiones=[
            ("Periodos de carencia", [
                "Las <b>intervenciones quirúrgicas</b> tienen una carencia de <b>seis (6) meses</b>.",
                "El <b>parto y la asistencia al embarazo</b> tienen una carencia de <b>diez (10) meses</b>.",
                "Los <b>tratamientos de reproducción asistida</b> tienen una carencia de <b>veinticuatro "
                "(24) meses</b> y se limitan a tres ciclos.",
            ]),
            ("Prestaciones no cubiertas", [
                "Las <b>enfermedades preexistentes</b> no declaradas en el cuestionario de salud.",
                "La <b>cirugía estética</b> y todo tratamiento sin finalidad terapéutica.",
                "Los tratamientos de <b>odontología</b> salvo extracción y limpieza anual.",
                "Los <b>medicamentos y prótesis</b> no incluidos en el catálogo.",
                "La asistencia <b>fuera del cuadro médico</b>, salvo urgencia vital acreditada.",
            ]),
        ],
        franquicias=[["Modalidad sin copago", "Sin franquicia", "Sin franquicia"],
                     ["Modalidad con copago", "Según acto médico", "10 € consulta / 25 € urgencia"],
                     ["Modalidad reembolso", "20% del gasto", "Límite 60.000 €/año"]],
    ),
]

os.makedirs("polizas", exist_ok=True)
for p in POLIZAS:
    construir_poliza(os.path.join("polizas", p["path"]), p["producto"], p["codigo"],
                     p["version"], p["coberturas"], p["exclusiones"], p["franquicias"])
print(f"✅ {len(POLIZAS)} pólizas generadas en ./polizas/")

✅ 3 pólizas generadas en ./polizas/


▶️ **Qué hace esta celda:** cuenta las páginas de cada PDF. **No es decorativo:** el Layout Parser tiene un límite de **15 páginas** por documento en modo online, así que conviene comprobarlo *antes* de gastar dinero procesando.

In [10]:
from pypdf import PdfReader

total = 0
for p in POLIZAS:
    ruta = os.path.join("polizas", p["path"])
    n = len(PdfReader(ruta).pages)
    total += n
    aviso = "⚠️ SUPERA EL LÍMITE" if n > 15 else "✓"
    print(f"{p['path']:24s} · {n:2d} págs · {os.path.getsize(ruta)/1024:5.0f} KB  {aviso}")

print(f"\n📄 Total: {total} páginas")
print(f"💶 Coste estimado de Document AI: {total/1000*10:.2f} $ ({total} págs × $10/1000)")

HOGAR_PLUS.pdf           ·  8 págs ·    12 KB  ✓
AUTO_TODO_RIESGO.pdf     ·  8 págs ·    12 KB  ✓
SALUD_FAMILIAR.pdf       ·  8 págs ·    12 KB  ✓

📄 Total: 24 páginas
💶 Coste estimado de Document AI: 0.24 $ (24 págs × $10/1000)


> 🚩 **CHECKPOINT 2** — Deberíais ver **3 pólizas de ~8 páginas** y un coste estimado de **~$0,24**. Si alguna supera 15 páginas, el parser la rechazará.

---
# 2 · Los PDF a Cloud Storage + object table ⏱️ ~10 min

### 🧠 Concepto clave — por qué los PDF NO van dentro de BigQuery

En el curso de emails cargábamos un DataFrame y ya. Aquí no podemos: **BigQuery es una base de datos analítica, no un almacén de binarios**. Los PDF viven en **Cloud Storage**.

Pero entonces, ¿cómo los "ve" SQL? Con una **object table**: una tabla externa donde **cada fila es un archivo** del bucket. No contiene el PDF: contiene su `uri`, su `content_type`, su tamaño… y una referencia firmada con la que **las funciones de IA de BigQuery pueden ir a leerlo** en Cloud Storage cuando lo necesiten.

> Es el mismo truco conceptual que el modelo remoto del curso anterior: **BigQuery no hace el trabajo, pero te deja pedirlo en SQL.**

▶️ **Qué hace esta celda:** crea el bucket y sube los PDF. El bucket debe estar en la **misma región** que el dataset.

In [11]:
!gcloud storage buckets create gs://{BUCKET} --location={LOCATION} --quiet 2>/dev/null || echo "(el bucket ya existía)"
!gcloud storage cp polizas/*.pdf gs://{BUCKET}/ --quiet

print("\n📦 Contenido del bucket:")
!gcloud storage ls -l gs://{BUCKET}/

Copying file://polizas/AUTO_TODO_RIESGO.pdf to gs://codecrypto-ai-polizas-rag/AUTO_TODO_RIESGO.pdf
Copying file://polizas/HOGAR_PLUS.pdf to gs://codecrypto-ai-polizas-rag/HOGAR_PLUS.pdf
Copying file://polizas/SALUD_FAMILIAR.pdf to gs://codecrypto-ai-polizas-rag/SALUD_FAMILIAR.pdf

Average throughput: 51.9kiB/s

📦 Contenido del bucket:
     12329  2026-07-21T17:03:32Z  gs://codecrypto-ai-polizas-rag/AUTO_TODO_RIESGO.pdf
     12401  2026-07-21T17:03:31Z  gs://codecrypto-ai-polizas-rag/HOGAR_PLUS.pdf
     12238  2026-07-21T17:03:32Z  gs://codecrypto-ai-polizas-rag/SALUD_FAMILIAR.pdf
TOTAL: 3 objects, 36968 bytes (36.10kiB)


▶️ **Qué hace esta celda:** los tres pasos administrativos, igual que en el curso anterior pero con **un rol más**. La conexión necesita permiso para: invocar Vertex AI (embeddings), **invocar Document AI** (el parser) y **leer el bucket** (los PDF).

> 🎓 Este es el error nº 1 en producción: la conexión existe, el SQL es correcto… y falla con `Permission denied` porque a su service account le falta uno de los tres roles.

In [12]:
import json, subprocess, time

CONN_ID = "vertex_conn"

# 1) Crear la conexión (idempotente)
!bq mk --connection --location={LOCATION} --project_id={PROJECT_ID} \
    --connection_type=CLOUD_RESOURCE {CONN_ID} 2>/dev/null || echo "(la conexión ya existía)"

# 2) Obtener su service account
out = subprocess.run(
    ["bq", "show", "--connection", "--format=json", f"{PROJECT_ID}.{LOCATION}.{CONN_ID}"],
    capture_output=True, text=True)
if out.returncode != 0:
    raise RuntimeError(f"'bq show' falló:\n{out.stderr}")
sa = json.loads(out.stdout[out.stdout.find("{"):])["cloudResource"]["serviceAccountId"]
print("Service account de la conexión:", sa)

# 3) Los TRES roles que necesita
for rol in ["roles/aiplatform.user",        # embeddings (Vertex AI)
            "roles/documentai.viewer",      # 👈 NUEVO: invocar el Layout Parser
            "roles/storage.objectViewer"]:  # 👈 NUEVO: leer los PDF del bucket
    r = subprocess.run(
        ["gcloud", "projects", "add-iam-policy-binding", PROJECT_ID,
         f"--member=serviceAccount:{sa}", f"--role={rol}",
         "--condition=None", "--quiet"],
        capture_output=True, text=True)
    print(f"{'✅' if r.returncode == 0 else '❌'} {rol}"
          f"{'' if r.returncode == 0 else '  → ' + r.stderr.strip()[:120]}")

Connection 540389217585.us.vertex_conn successfully created
Service account de la conexión: bqcx-540389217585-966s@gcp-sa-bigquery-condel.iam.gserviceaccount.com
✅ roles/aiplatform.user
✅ roles/documentai.viewer
✅ roles/storage.objectViewer


▶️ **Qué hace esta celda:** crea la object table sobre los PDF del bucket. Fíjate en `object_metadata = 'SIMPLE'`: es lo que le dice a BigQuery *"cada fila es un objeto"*.

In [13]:
sql_object_table = f"""
CREATE OR REPLACE EXTERNAL TABLE `{PROJECT_ID}.{DATASET}.polizas_pdf`
WITH CONNECTION `{PROJECT_ID}.{LOCATION.lower()}.{CONN_ID}`
OPTIONS (
  object_metadata = 'SIMPLE',
  uris = ['gs://{BUCKET}/*.pdf']
)
"""
client.query(sql_object_table).result()

# Una object table se consulta como cualquier tabla... pero sus filas son ARCHIVOS
client.query(f"""
  SELECT uri, content_type, ROUND(size/1024, 1) AS kb, updated
  FROM `{PROJECT_ID}.{DATASET}.polizas_pdf`
  ORDER BY uri
""").to_dataframe()

,uri,content_type,kb,updated
0,gs://codecrypto-ai-polizas-rag/AUTO_TODO_RIESG...,application/pdf,12.0,2026-07-21 17:03:32.194000+00:00
1,gs://codecrypto-ai-polizas-rag/HOGAR_PLUS.pdf,application/pdf,12.1,2026-07-21 17:03:31.484000+00:00
2,gs://codecrypto-ai-polizas-rag/SALUD_FAMILIAR.pdf,application/pdf,12.0,2026-07-21 17:03:32.174000+00:00


> 🚩 **CHECKPOINT 3** — Deberíais ver **3 filas**, una por PDF, con su `uri` y `content_type = application/pdf`. Si salen 0 filas, el `uris` del `CREATE` no coincide con lo que hay en el bucket.

---
# 3 · Document AI: procesador y modelo remoto ⏱️ ~10 min

Necesitamos dos piezas:

1. **Un procesador** de Document AI del tipo `LAYOUT_PARSER_PROCESSOR`. Es el "motor" que entiende páginas.
2. **Un modelo remoto** en BigQuery que apunte a ese procesador — exactamente el mismo patrón que el `embedding_model` del curso anterior.

> ⚠️ **Curiosidad operativa:** `gcloud` **no tiene** comandos para Document AI. Hay que llamar a la **REST API** a pelo. Es un buen recordatorio de que la CLI no siempre cubre todo GCP.

▶️ **Qué hace esta celda:** crea el procesador vía REST (o reutiliza el que ya exista) y se queda con su **ID desnudo** — no la ruta completa, porque es lo que espera BigQuery en el paso siguiente.

In [14]:
import requests, subprocess, json

def _token():
    return subprocess.run(["gcloud", "auth", "print-access-token"],
                          capture_output=True, text=True).stdout.strip()

API = f"https://{LOC_DOCAI}-documentai.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOC_DOCAI}/processors"
HDRS = {"Authorization": f"Bearer {_token()}", "Content-Type": "application/json; charset=utf-8"}

# ¿Ya existe uno nuestro? (idempotencia: no queremos crear uno en cada ejecución)
existentes = requests.get(API, headers=HDRS).json().get("processors", [])
mio = next((p for p in existentes
            if p.get("displayName") == "polizas-layout-parser"
            and p.get("type") == "LAYOUT_PARSER_PROCESSOR"), None)

if mio is None:
    resp = requests.post(API, headers=HDRS, json={
        "type": "LAYOUT_PARSER_PROCESSOR",
        "displayName": "polizas-layout-parser",
    })
    if resp.status_code >= 300:
        raise RuntimeError(f"No se pudo crear el procesador ({resp.status_code}):\n{resp.text}\n"
                           f"👉 Causa típica: facturación no activa en el proyecto.")
    mio = resp.json()
    print("✅ Procesador creado")
else:
    print("♻️  Reutilizamos el procesador existente")

# BigQuery quiere el ID pelado, no 'projects/.../processors/XXXX'
PROCESSOR_ID = mio["name"].split("/")[-1]
print("   name :", mio["name"])
print("   ID   :", PROCESSOR_ID, "  👈 esto es lo que usa BigQuery")
print("   state:", mio.get("state"))

✅ Procesador creado
   name : projects/540389217585/locations/us/processors/b9a77c997c85a60d
   ID   : b9a77c997c85a60d   👈 esto es lo que usa BigQuery
   state: ENABLED


▶️ **Qué hace esta celda:** registra el modelo remoto que conecta BigQuery ↔ Document AI. Compáralo con el `embedding_model` del curso anterior: **mismo patrón**, distinto `REMOTE_SERVICE_TYPE`.

In [15]:
sql_modelo_docai = f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.{DATASET}.layout_parser`
REMOTE WITH CONNECTION `{PROJECT_ID}.{LOCATION.lower()}.{CONN_ID}`
OPTIONS (
  REMOTE_SERVICE_TYPE = 'CLOUD_AI_DOCUMENT_V1',
  DOCUMENT_PROCESSOR = '{PROCESSOR_ID}'
)
"""
for intento in range(1, 5):
    try:
        client.query(sql_modelo_docai).result()
        print("✅ Modelo remoto 'layout_parser' creado")
        break
    except Exception as e:
        if intento == 4:
            raise
        print(f"⏳ Permisos propagándose (intento {intento}/4). Reintento en 30 s...")
        time.sleep(30)

✅ Modelo remoto 'layout_parser' creado


---
# 4 · `ML.PROCESS_DOCUMENT`: del PDF a chunks con estructura ⏱️ ~15 min

Este es **el momento clave del curso**. Una sola función SQL convierte 3 PDF binarios en fragmentos de texto que **saben de dónde vienen**.

Las dos opciones que le pasamos merecen atención:

| Opción | Qué hace | Por qué la elegimos |
|---|---|---|
| `chunk_size: 250` | tamaño objetivo del chunk **en tokens** | equilibrio: cabe una cláusula entera sin diluir el significado |
| `include_ancestor_headings: true` | 🎯 **antepone a cada chunk los títulos de los que cuelga** | **es la línea que salva el curso**: sin ella, el chunk de exclusiones llega huérfano |

> 🧠 Compara con el curso de emails: allí `trocear()` era una función Python nuestra que cortaba por caracteres. Aquí **el troceado lo hace un modelo que ha leído la página**.

▶️ **Qué hace esta celda:** procesa los 3 PDF. Tarda **1-3 minutos** (cada página pasa por el parser). El `WHERE ... status = ''` no es paranoia: la documentación avisa de que **un job puede terminar "OK" con filas fallidas dentro**.

In [18]:
json_process_options = '{"layout_config":{"chunking_config":{"chunk_size":250,"include_ancestor_headings":true}}}'
sql_process = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET}.polizas_procesadas` AS
SELECT *
FROM ML.PROCESS_DOCUMENT(
  MODEL `{PROJECT_ID}.{DATASET}.layout_parser`,
  TABLE `{PROJECT_ID}.{DATASET}.polizas_pdf`,
  PROCESS_OPTIONS => (JSON '{json_process_options}')
)
"""
print("⏳ Procesando los PDF con Document AI (1-3 min)...")
client.query(sql_process).result()

# 🛡️ Auditoría: ¿falló alguna fila silenciosamente?
client.query(f"""
  SELECT
    uri,
    IF(ml_process_document_status = '', '✅ OK', ml_process_document_status) AS estado
  FROM `{PROJECT_ID}.{DATASET}.polizas_procesadas`
  ORDER BY uri
""").to_dataframe()

⏳ Procesando los PDF con Document AI (1-3 min)...


,uri,estado
0,gs://codecrypto-ai-polizas-rag/AUTO_TODO_RIESG...,✅ OK
1,gs://codecrypto-ai-polizas-rag/HOGAR_PLUS.pdf,✅ OK
2,gs://codecrypto-ai-polizas-rag/SALUD_FAMILIAR.pdf,✅ OK


▶️ **Qué hace esta celda:** desempaqueta el JSON. El parser devuelve **un JSON gigante por documento**; aquí lo convertimos en **una fila por chunk**, con sus metadatos.

Los caminos JSON (`$.chunkId`, `$.pageSpan.pageStart`…) salen del formato `chunkedDocument` de Document AI. Fíjate en lo que estamos **fabricando**: `pagina_inicio` es, literalmente, **la capacidad de citar**.

In [ ]:
sql_chunks = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET}.polizas_chunks` AS
SELECT
  REGEXP_EXTRACT(uri, r'([^/]+)\\.pdf$')                        AS documento,
  uri,
  JSON_EXTRACT_SCALAR(chunk, '$.chunkId')                        AS chunk_id,
  JSON_EXTRACT_SCALAR(chunk, '$.content')                        AS content,
  CAST(JSON_EXTRACT_SCALAR(chunk, '$.pageSpan.pageStart') AS INT64) AS pagina_inicio,
  CAST(JSON_EXTRACT_SCALAR(chunk, '$.pageSpan.pageEnd')   AS INT64) AS pagina_fin,
  -- 🎯 el heading ancestro viene inyectado al principio del content, en Markdown:
  --    lo extraemos a su propia columna para poder auditarlo y filtrar por él
  REGEXP_EXTRACT(JSON_EXTRACT_SCALAR(chunk, '$.content'), r'(?m)^#{{1,6}}\\s+(.+)$') AS heading
FROM `{PROJECT_ID}.{DATASET}.polizas_procesadas`,
     UNNEST(JSON_EXTRACT_ARRAY(ml_process_document_result.chunkedDocument.chunks, '$')) AS chunk
WHERE ml_process_document_status = ''
"""
client.query(sql_chunks).result()

client.query(f"""
  SELECT documento, COUNT(*) AS chunks,
         MIN(pagina_inicio) AS pag_min, MAX(pagina_fin) AS pag_max
  FROM `{PROJECT_ID}.{DATASET}.polizas_chunks`
  GROUP BY documento ORDER BY documento
""").to_dataframe()

▶️ **Qué hace esta celda:** el momento de la verdad. Buscamos los chunks que hablan de **agua** en Hogar Plus y miramos **qué heading llevan pegado**.

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 90)

client.query(f"""
  SELECT pagina_inicio AS pag, heading, SUBSTR(REGEXP_REPLACE(content, r'\\s+', ' '), 1, 150) AS extracto
  FROM `{PROJECT_ID}.{DATASET}.polizas_chunks`
  WHERE documento = 'HOGAR_PLUS' AND LOWER(content) LIKE '%agua%'
  ORDER BY pagina_inicio
""").to_dataframe()

> 🚩 **CHECKPOINT 4 + ☕ pausa** — Mira bien esa tabla. Deberías ver chunks sobre "agua" en **dos páginas distintas**, y —lo importante— **con headings diferentes**: unos cuelgan de `3. COBERTURAS` y otros de `4. EXCLUSIONES`.
>
> 🎓 **Eso es exactamente lo que compramos con `include_ancestor_headings`.** El chunk ya no es un párrafo suelto: es un párrafo **que sabe bajo qué título vive**. Cuando dentro de un rato se lo pasemos a Gemini, el modelo verá ese título y responderá bien.

---
# 5 · El contraste: ¿y si lo hubiéramos hecho "a la manera de los emails"? ⏱️ ~10 min

Toca justificar el gasto. Acabamos de pagar $0,24 y montar dos servicios. ¿No habría bastado con `pypdf` y la función `trocear()` del curso anterior?

Vamos a hacerlo **de verdad** y comparar. Esta es la celda más importante del curso para entender **por qué** existe el resto.

▶️ **Qué hace esta celda:** replica el enfoque del curso de emails sobre el PDF: extraer todo el texto con `pypdf` y cortarlo cada 700 caracteres, sin mirar la estructura. Luego busca el trozo que habla de lluvia y lo muestra **tal y como le llegaría al LLM**.

In [ ]:
# ═══ Reproducimos el chunking "naive" del curso de emails ═══
reader = PdfReader("polizas/HOGAR_PLUS.pdf")
texto_plano = "\n".join(p.extract_text() or "" for p in reader.pages)

def trocear(texto, tam=700, solapa=100):
    """La misma idea del curso anterior: cortar por caracteres."""
    trozos, i = [], 0
    while i < len(texto):
        trozos.append(texto[i:i+tam])
        i += tam - solapa
    return trozos

trozos = trocear(texto_plano)
print(f"pypdf + corte por caracteres → {len(trozos)} trozos\n")

# ¿Qué trozo habla de lluvia? Es el que recuperaría el buscador.
culpable = next(t for t in trozos if "lluvia" in t.lower())

print("═" * 70)
print("ESTO ES LO QUE LE LLEGARÍA AL LLM (chunking naive):")
print("═" * 70)
print(culpable.strip()[:700])
print("═" * 70)
print("\n❓ Pregúntate: leyendo SOLO esto, ¿sabrías decir si está cubierto o excluido?")
print("❓ ¿Ves por algún lado el título '4. EXCLUSIONES'? ¿Y en qué página estabas?")

▶️ **Qué hace esta celda:** y ahora lo mismo, pero con el chunk que produjo el Layout Parser. **Compara los dos bloques de texto.**

In [ ]:
df_bueno = client.query(f"""
  SELECT heading, pagina_inicio, content
  FROM `{PROJECT_ID}.{DATASET}.polizas_chunks`
  WHERE documento = 'HOGAR_PLUS' AND LOWER(content) LIKE '%lluvia%'
  LIMIT 1
""").to_dataframe()

print("═" * 70)
print("ESTO ES LO QUE LE LLEGA AL LLM (Layout Parser):")
print("═" * 70)
print(df_bueno.content.iloc[0][:700])
print("═" * 70)
print(f"\n📍 Y además, en columnas aparte: heading = {df_bueno.heading.iloc[0]!r}"
      f" | página = {df_bueno.pagina_inicio.iloc[0]}")
print("\n✅ El chunk se defiende solo: lleva su título, y sabemos dónde citarlo.")

### 🧭 Para llevar (bloque 5) — el resumen del curso en cuatro líneas

| | Chunking naive (`pypdf` + cortar) | Layout Parser |
|---|---|---|
| **Jerarquía** | ❌ se pierde: el chunk queda huérfano | ✅ inyectada en el chunk |
| **Tablas** | ❌ espagueti: cifras sin su fila | ✅ se respetan como bloque |
| **Página** | ❌ desconocida → **no puedes citar** | ✅ `pageSpan` |
| **Escaneados** | ❌ devuelve vacío | ✅ OCR incluido |
| **Coste** | gratis | $10 / 1.000 págs |

> 🎓 **La conclusión no es "usa siempre Document AI".** Es: **el chunking correcto depende de la estructura del documento**. Para emails, cortar por caracteres estaba bien: no había jerarquía que perder. Para un condicionado con secciones que se contradicen entre sí, **cortar por caracteres es un bug con apariencia de funcionar**.
>
> Y ese es el peor tipo de bug: el que responde con seguridad y se equivoca.

---
# 6 · Embeddings y búsqueda ⏱️ ~10 min

**Aquí no hay nada nuevo.** A partir de este punto el pipeline es idéntico al del curso de emails: los chunks ya son texto en una tabla normal. Vamos rápido.

> 🎓 **Recordatorio exprés** — *vectorizar* es convertir cada chunk en un vector que captura su significado (`ML.GENERATE_EMBEDDING`). El `task_type` le dice al modelo el propósito del texto: **`RETRIEVAL_DOCUMENT`** al vectorizar los chunks del corpus, **`RETRIEVAL_QUERY`** al vectorizar la pregunta. Y *buscar* (`VECTOR_SEARCH`) es quedarse con los chunks cuyo vector está más **cerca** del de la pregunta. Si quieres el porqué a fondo, está en el curso de emails; para seguir aquí, con esto basta.

▶️ **Qué hace esta celda:** registra el modelo de embeddings (idéntico al curso anterior) y vectoriza los chunks.

In [ ]:
# Modelo de embeddings — exactamente el mismo del curso de emails
client.query(f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.{DATASET}.embedding_model`
REMOTE WITH CONNECTION `{PROJECT_ID}.{LOCATION.lower()}.{CONN_ID}`
OPTIONS (ENDPOINT = 'gemini-embedding-001')
""").result()
print("✅ Modelo de embeddings creado")

sql_emb = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET}.polizas_embeddings` AS
SELECT *
FROM ML.GENERATE_EMBEDDING(
  MODEL `{PROJECT_ID}.{DATASET}.embedding_model`,
  (SELECT chunk_id, documento, pagina_inicio, pagina_fin, heading, content
   FROM `{PROJECT_ID}.{DATASET}.polizas_chunks`),
  STRUCT(TRUE AS flatten_json_output, 'RETRIEVAL_DOCUMENT' AS task_type)
)
WHERE ml_generate_embedding_status = ''
"""
print("⏳ Vectorizando...")
client.query(sql_emb).result()

client.query(f"""
  SELECT (SELECT COUNT(*) FROM `{PROJECT_ID}.{DATASET}.polizas_chunks`) AS chunks_totales,
         COUNT(*) AS vectorizados,
         ARRAY_LENGTH(ANY_VALUE(ml_generate_embedding_result)) AS dimensiones
  FROM `{PROJECT_ID}.{DATASET}.polizas_embeddings`
""").to_dataframe()

▶️ **Qué hace esta celda:** la función de búsqueda. Igual que `buscar_emails()`, pero **devolviendo lo que hace falta para citar**: documento, página y heading. Además acepta un filtro por producto (**búsqueda híbrida**: semántica + `WHERE`).

In [ ]:
def buscar_clausulas(pregunta: str, k: int = 5, documento: str | None = None):
    """Devuelve los k chunks más relevantes, con su procedencia para poder citarlos."""
    filtro = "AND base.documento = @documento" if documento else ""
    sql = f"""
    SELECT base.documento, base.pagina_inicio AS pagina, base.heading,
           base.content, ROUND(distance, 4) AS distancia
    FROM VECTOR_SEARCH(
      TABLE `{PROJECT_ID}.{DATASET}.polizas_embeddings`,
      'ml_generate_embedding_result',
      (SELECT ml_generate_embedding_result
       FROM ML.GENERATE_EMBEDDING(
         MODEL `{PROJECT_ID}.{DATASET}.embedding_model`,
         (SELECT @pregunta AS content),
         STRUCT(TRUE AS flatten_json_output, 'RETRIEVAL_QUERY' AS task_type))),
      top_k => @k, distance_type => 'COSINE')
    WHERE TRUE {filtro}
    ORDER BY distance
    """
    params = [bigquery.ScalarQueryParameter("pregunta", "STRING", pregunta),
              bigquery.ScalarQueryParameter("k", "INT64", k)]
    if documento:
        params.append(bigquery.ScalarQueryParameter("documento", "STRING", documento))
    return client.query(sql, job_config=bigquery.QueryJobConfig(query_parameters=params)).to_dataframe()

# 🎯 La pregunta del millón. Mira los HEADINGS de lo que recupera:
buscar_clausulas("¿me cubre el agua de lluvia que ha entrado en casa?",
                 k=4, documento="HOGAR_PLUS")[["documento", "pagina", "heading", "distancia"]]

> 🎓 **Observa el resultado:** el buscador recupera chunks de **coberturas y de exclusiones a la vez**, con distancias parecidas. Y hace bien: semánticamente **los dos hablan de daños por agua**.
>
> Esto demuestra que **el retrieval por sí solo no puede resolver este problema**. La distinción cobertura/exclusión no está en el *parecido semántico* — está en **la jerarquía**. Por eso el heading tiene que llegar hasta el LLM.

---
# 7 · RAG con citas verificables ⏱️ ~15 min

En el curso de emails pedíamos al modelo que citara los `message_id`. Aquí la exigencia sube de nivel: **póliza, página y cláusula**. Un asesor tiene que poder **abrir el PDF y comprobarlo**.

> 🔁 **El patrón RAG, por si es tu primera vez** — tres pasos: **recuperar** los fragmentos relevantes con `VECTOR_SEARCH` → **aumentar** el prompt pegándoselos como contexto → **generar** la respuesta con Gemini a partir de *solo* ese contexto. La función `responder()` de abajo es exactamente esos tres pasos, uno detrás de otro.

Dos cosas nuevas en el prompt, y las dos son deliberadas:

1. **Le damos el heading explícitamente** en cada fragmento del contexto.
2. **Le obligamos a distinguir** cobertura de exclusión, y a mirar el título antes de responder.

> 🧠 Un RAG sin citas es un oráculo: te da una respuesta y te pide fe. Un RAG con citas es una **herramienta de trabajo**: te da una respuesta *y dónde verificarla*. En seguros, la diferencia es legal.

▶️ **Qué hace esta celda:** el RAG completo. Compáralo con `responder()` del curso anterior: la estructura es idéntica (recuperar → montar contexto → preguntar), pero el contexto ahora **arrastra la procedencia** y el prompt **exige citarla**.

In [ ]:
from google import genai

genai_client = genai.Client(vertexai=True, project=PROJECT_ID, location="global")

def responder(pregunta: str, k: int = 6, documento: str | None = None, verbose: bool = True) -> str:
    # 1) Retrieval
    docs = buscar_clausulas(pregunta, k, documento)

    # 2) Augmentation — cada fragmento viaja CON su procedencia
    contexto = "\n\n---\n\n".join(
        f"[Póliza: {r.documento} | Página: {r.pagina} | Sección: {r.heading}]\n{r.content}"
        for r in docs.itertuples()
    )
    prompt = f"""Eres el asistente del equipo de atención al cliente de Peñalara Seguros, S.A.
Respondes sobre lo que cubre o no cubre una póliza, usando EXCLUSIVAMENTE los fragmentos del contexto.

REGLAS OBLIGATORIAS:
1. Antes de afirmar que algo está cubierto, comprueba la SECCIÓN del fragmento.
   Un fragmento bajo "EXCLUSIONES" significa que NO está cubierto, aunque hable del mismo riesgo.
2. Si el mismo riesgo aparece en coberturas Y en exclusiones, explica el MATIZ que los separa
   (normalmente la causa del daño distingue un caso del otro).
3. Cita SIEMPRE la póliza, la página y la cláusula concreta. Formato: (Hogar Plus, pág. 5, cláusula 4.1.a)
4. Si el contexto no basta para responder, dilo claramente. NO inventes coberturas.

### Contexto (fragmentos recuperados del condicionado):
{contexto}

### Pregunta del cliente:
{pregunta}"""

    # 3) Generation
    resp = genai_client.models.generate_content(model="gemini-2.5-flash", contents=prompt)
    if verbose:
        secciones = [s for s in docs.heading.unique().tolist() if s]
        print(f"🔎 {len(docs)} fragmentos | secciones tocadas: {secciones}\n")
    return resp.text or "(respuesta vacía o bloqueada por el filtro de seguridad — reintenta)"

# 🎯 LA PREGUNTA TRAMPA — la que un RAG naive responde mal
print(responder("Ha entrado agua de lluvia por la terraza y se me ha estropeado el parqué. "
                "¿Me lo cubre el seguro de hogar?", documento="HOGAR_PLUS"))

> 🎓 **El momento del curso.** Si todo ha ido bien, el modelo te habrá dicho que **NO está cubierto**, citando la cláusula `4.1.a` (filtraciones a través de terrazas, aun por lluvia) y probablemente habrá explicado el matiz: lo que sí se cubre es la **rotura accidental de conducciones** (cláusula 3.2), no la filtración.
>
> **Ha distinguido dos textos casi idénticos porque uno venía con `4. EXCLUSIONES` pegado en la frente.** Eso lo hemos comprado en el bloque 4 con una sola opción del parser.

▶️ **Qué hacen estas dos celdas:** más preguntas, elegidas para tocar otros documentos y otros matices (carencias en salud, exclusiones de auto).

In [ ]:
print(responder("Contraté el seguro de salud hace 3 meses y necesito operarme del menisco. "
                "¿Me lo cubren ya?", documento="SALUD_FAMILIAR"))

In [ ]:
print(responder("Tuve un accidente y el coche lo conducía mi sobrino, que sacó el carné hace 8 meses "
                "y no está declarado en la póliza. ¿Qué franquicia me toca pagar?",
                documento="AUTO_TODO_RIESGO"))

> 🎓 **Anti-alucinación en acción:** pregunta algo que **no esté** en ningún condicionado, por ejemplo
> `responder("¿Cubre la póliza los daños causados por un ataque de dragones?")`.
> Con las reglas del prompt, el modelo debería reconocer que el contexto no lo contempla en vez de inventarse una cláusula. **Pruébalo**: es la mejor forma de ganar confianza en un RAG.

---
# 8 · Versionado de documentos: el problema que los emails no tenían ⏱️ ~10 min

En el curso anterior la ingesta incremental era sencilla: **llegan emails nuevos, se añaden**. Un email nunca se corrige a sí mismo.

**Los documentos sí.** Peñalara publica *Hogar Plus v3.3* y esa versión **sustituye** a la v3.2. Si te limitas a añadir los chunks nuevos, tu RAG acaba con **dos versiones contradictorias del mismo condicionado** — y citará la que le apetezca. Es el fallo silencioso más común de los RAG documentales en producción.

> 🧠 **Concepto clave — el documento es la unidad de reemplazo.** En emails, la unidad era la fila y sólo añadías. En documentos, cuando llega una versión nueva hay que **retirar la anterior entera**: borrar sus chunks y sus vectores, no mezclarlos.

▶️ **Qué hace esta celda:** simula la publicación de **Hogar Plus v3.3**, que sube la franquicia de daños por agua de 150 € a 250 €. Procesamos solo el PDF nuevo, y **eliminamos los chunks de la versión anterior antes de insertar los nuevos** (`DELETE` + `INSERT` por documento).

In [ ]:
# 1) Nueva versión del condicionado: cambia la franquicia de daños por agua
nueva = dict(POLIZAS[0])          # copia de Hogar Plus
nueva["version"] = "3.3"
nueva["coberturas"] = [c[:] for c in POLIZAS[0]["coberturas"]]
nueva["coberturas"][1][2] = "250 €"     # 👈 150 € → 250 €

construir_poliza("polizas/HOGAR_PLUS.pdf", nueva["producto"], nueva["codigo"],
                 nueva["version"], nueva["coberturas"], nueva["exclusiones"], nueva["franquicias"])
!gcloud storage cp polizas/HOGAR_PLUS.pdf gs://{BUCKET}/ --quiet
print("✅ Hogar Plus v3.3 publicada y subida")

# 2) Reprocesamos SOLO ese PDF (no los tres: cada página cuesta dinero)
client.query(f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET}.polizas_procesadas_delta` AS
SELECT * FROM ML.PROCESS_DOCUMENT(
  MODEL `{PROJECT_ID}.{DATASET}.layout_parser`,
  (SELECT uri, content_type FROM `{PROJECT_ID}.{DATASET}.polizas_pdf`
   WHERE uri LIKE '%HOGAR_PLUS.pdf'),
  PROCESS_OPTIONS => (JSON '{{"layout_config":{{"chunking_config":{{"chunk_size":250,"include_ancestor_headings":true}}}}}}')
)
""").result()

# 3) 🔑 Fuera la versión vieja, entra la nueva (atómico por documento)
client.query(f"""
DELETE FROM `{PROJECT_ID}.{DATASET}.polizas_chunks` WHERE documento = 'HOGAR_PLUS'
""").result()
client.query(f"""
INSERT INTO `{PROJECT_ID}.{DATASET}.polizas_chunks`
SELECT
  REGEXP_EXTRACT(uri, r'([^/]+)\\.pdf$'), uri,
  JSON_EXTRACT_SCALAR(chunk, '$.chunkId'),
  JSON_EXTRACT_SCALAR(chunk, '$.content'),
  CAST(JSON_EXTRACT_SCALAR(chunk, '$.pageSpan.pageStart') AS INT64),
  CAST(JSON_EXTRACT_SCALAR(chunk, '$.pageSpan.pageEnd') AS INT64),
  REGEXP_EXTRACT(JSON_EXTRACT_SCALAR(chunk, '$.content'), r'(?m)^#{{1,6}}\\s+(.+)$')
FROM `{PROJECT_ID}.{DATASET}.polizas_procesadas_delta`,
     UNNEST(JSON_EXTRACT_ARRAY(ml_process_document_result.chunkedDocument.chunks, '$')) AS chunk
WHERE ml_process_document_status = ''
""").result()

# 4) Lo mismo con los vectores: fuera los viejos, vectorizamos solo los nuevos
client.query(f"""
DELETE FROM `{PROJECT_ID}.{DATASET}.polizas_embeddings` WHERE documento = 'HOGAR_PLUS'
""").result()
job = client.query(f"""
INSERT INTO `{PROJECT_ID}.{DATASET}.polizas_embeddings`
SELECT * FROM ML.GENERATE_EMBEDDING(
  MODEL `{PROJECT_ID}.{DATASET}.embedding_model`,
  (SELECT chunk_id, documento, pagina_inicio, pagina_fin, heading, content
   FROM `{PROJECT_ID}.{DATASET}.polizas_chunks` WHERE documento = 'HOGAR_PLUS'),
  STRUCT(TRUE AS flatten_json_output, 'RETRIEVAL_DOCUMENT' AS task_type))
WHERE ml_generate_embedding_status = ''
""")
job.result()
print(f"✅ Reemplazados {job.num_dml_affected_rows} chunks de Hogar Plus (v3.2 → v3.3)")

▶️ **Qué hace esta celda:** comprueba que el RAG ya responde con la franquicia **nueva** y que no ha quedado ningún rastro de la vieja.

In [ ]:
print(responder("¿Cuál es la franquicia por daños por agua en el seguro de hogar?",
                documento="HOGAR_PLUS"))

print("\n" + "═"*70)
print("🔍 ¿Ha quedado algún chunk duplicado de la versión anterior?")
client.query(f"""
  SELECT documento, COUNT(*) AS chunks, COUNT(DISTINCT chunk_id) AS ids_unicos
  FROM `{PROJECT_ID}.{DATASET}.polizas_embeddings`
  GROUP BY documento ORDER BY documento
""").to_dataframe()

> 🎓 Debería responder **250 €**. Si respondiera 150 €, o dudara entre las dos cifras, tendrías el bug clásico: **dos versiones conviviendo**. Por eso el `DELETE` por documento va **antes** del `INSERT`, y por eso la columna `documento` es la clave de todo este bloque.

---
# 9 · Evaluación, ejercicios y limpieza ⏱️ ~10 min

### Mini-evaluación: ¿sabe distinguir cobertura de exclusión?

Para un RAG de emails evaluábamos si recuperaba la categoría correcta. Aquí lo que importa es más fino: **¿acierta el sentido de la respuesta?**

▶️ **Qué hace esta celda:** un test con casos donde la respuesta correcta se conoce de antemano. Cada uno está elegido porque el retrieval recupera fragmentos de coberturas *y* de exclusiones — y sólo la jerarquía desempata.

In [ ]:
casos = [
    ("Se ha roto una tubería del baño y ha inundado el salón. ¿Está cubierto?",
     "HOGAR_PLUS", "SÍ (cláusula 3.2, rotura accidental de conducciones)"),
    ("Entra agua de lluvia por una filtración en la terraza. ¿Está cubierto?",
     "HOGAR_PLUS", "NO (cláusula 4.1.a, filtraciones aunque sean por lluvia)"),
    ("Dejé la ventana abierta, llovió y se estropeó el suelo. ¿Está cubierto?",
     "HOGAR_PLUS", "NO (cláusula 4.1.d, agua por huecos dejados abiertos)"),
    ("Necesito una operación de rodilla a los 8 meses de contratar. ¿Cubierta?",
     "SALUD_FAMILIAR", "SÍ (carencia quirúrgica de 6 meses ya superada)"),
    ("Quiero una rinoplastia estética. ¿La cubre el seguro de salud?",
     "SALUD_FAMILIAR", "NO (exclusión: cirugía estética sin fin terapéutico)"),
]

for pregunta, doc, esperado in casos:
    print("═" * 78)
    print(f"❓ {pregunta}")
    print(f"🎯 Esperado: {esperado}")
    print(f"🤖 {responder(pregunta, documento=doc, verbose=False)[:300]}...\n")

### 🧪 Autoevaluación — cinco preguntas, cero código

1. ¿Por qué un PDF necesita `ML.PROCESS_DOCUMENT` y un email no?
2. ¿Qué pasaría exactamente si pusiéramos `include_ancestor_headings: false`? ¿En qué pregunta concreta se rompería el sistema?
3. ¿Por qué los PDF están en Cloud Storage y no en una columna `BYTES` de BigQuery?
4. La ingesta incremental de emails era `INSERT`. ¿Por qué la de pólizas necesita `DELETE` + `INSERT`?
5. Tu retrieval recupera el chunk correcto y aun así el RAG responde mal. ¿Dónde miras primero?

### Ejercicios para casa

- **Fácil** — Añade una cuarta póliza (*Comercio Protegido*) y comprueba que el RAG la incorpora sin tocar el pipeline. *Pista: es solo un dict más en `POLIZAS`.*
- **Medio** — Haz que `responder()` devuelva también un **enlace al PDF en la página citada**. *Pista: los visores aceptan `...pdf#page=5`; combínalo con una signed URL del bucket.*
- **Medio** — Un cliente pregunta sin decir de qué póliza habla. Haz que el sistema **decida el documento** antes de filtrar. *Pista: una primera llamada al LLM clasificando la pregunta, o buscar sin filtro y quedarte con el `documento` dominante.*
- **Difícil** — **Rompe el sistema a propósito**: reprocesa con `chunk_size: 50` y busca una pregunta que antes acertaba y ahora falla. Explica por qué. *Pista: mira qué le pasa a las tablas de coberturas.*
- **Difícil** — Convierte una póliza en **PDF escaneado** (rasteriza sus páginas a imagen) y pásala por el mismo pipeline. Comprueba que el Layout Parser sigue funcionando y que `pypdf` devuelve vacío. *Ahí verás el OCR ganándose el sueldo.*

### Ideas para producción

- **Alertas de coste**: cada reproceso son páginas facturadas. Un `DELETE`+`INSERT` accidental sobre 10.000 PDF duele.
- **`md5_hash` como disparador**: la object table lo expone → reprocesa un documento **solo si su hash cambió**.
- **Guarda la versión** del condicionado como columna: permite responder *"según la póliza vigente en la fecha del siniestro"*, que es lo que realmente pregunta un perito.
- **Evaluación continua**: el juego de casos de arriba, en CI. Si un cambio de `chunk_size` baja el acierto, te enteras antes de desplegarlo.

### 🧹 Limpieza de recursos

> ⚠️ Esto borra el dataset, el bucket **y el procesador de Document AI**. El procesador no cuesta por existir (solo por página procesada), pero es buena higiene.

In [ ]:
BORRAR = False  # ⚠️ cambia a True para borrar todo

if BORRAR:
    client.delete_dataset(f"{PROJECT_ID}.{DATASET}", delete_contents=True, not_found_ok=True)
    print("🗑️  Dataset borrado")

    r = subprocess.run(["gcloud", "storage", "rm", "-r", f"gs://{BUCKET}", "--quiet"],
                       capture_output=True, text=True)
    print("🗑️  Bucket borrado" if r.returncode == 0 else "ℹ️  Bucket ya no existía")

    r = requests.delete(f"{API}/{PROCESSOR_ID}", headers={"Authorization": f"Bearer {_token()}"})
    print("🗑️  Procesador borrado" if r.status_code < 300 else f"⚠️  {r.status_code}: {r.text[:200]}")

    r = subprocess.run(["bq", "rm", "--connection", "--force",
                        f"{PROJECT_ID}.{LOCATION}.{CONN_ID}"], capture_output=True, text=True)
    print("🗑️  Conexión borrada" if r.returncode == 0 else "ℹ️  Conexión ya no existía")
else:
    print("ℹ️  BORRAR = False. Cambia a True y re-ejecuta para limpiar.")
    print("   Recuerda: el bucket y el dataset sí generan coste de almacenamiento (céntimos).")

---
## 🧭 Mapa final: qué te llevas, más allá de la tecnología

Si dentro de dos años Document AI se llama de otra forma, esto sigue siendo verdad:

1. **Un PDF no es texto: es un dibujo de un texto.** Todo RAG documental empieza por reconstruir lo que el formato tiró a la basura.
2. **Un chunk sin su contexto jerárquico es una trampa**, no información. Y es una trampa que *funciona el 90% de las veces*, que es lo que la hace peligrosa.
3. **La procedencia (documento, página, cláusula) no es un extra: es el producto.** En dominios regulados, una respuesta sin cita no vale nada porque no se puede verificar.
4. **El chunking correcto depende de la estructura de la fuente.** No hay un `chunk_size` universal; hay documentos que se dejan cortar y documentos que no.
5. **Los documentos se versionan; los emails no.** Toda ingesta documental necesita una estrategia de reemplazo, o acabarás citando la póliza del año pasado.

### El mismo curso en las otras nubes

| | Google Cloud | AWS | Azure |
|---|---|---|---|
| Extracción con layout | **Document AI Layout Parser** | Textract (Layout) | Document Intelligence |
| Almacén de binarios | Cloud Storage | S3 | Blob Storage |
| Embeddings + búsqueda | BigQuery + Vertex AI | Bedrock + OpenSearch | Azure AI Search |

Cambian los nombres. **Los cinco puntos de arriba, no.**

### 📚 Para seguir

- [Document AI Layout Parser](https://cloud.google.com/document-ai/docs/layout-parse-chunk)
- [`ML.PROCESS_DOCUMENT`](https://cloud.google.com/bigquery/docs/process-document)
- [Object tables](https://cloud.google.com/bigquery/docs/object-tables)
- Curso anterior: [*De la bandeja de entrada al RAG*](https://colab.research.google.com/github/noelserdna/colab-gcp-ia/blob/main/curso_rag_emails_bigquery_v2.ipynb)
